# Notebook 3 Vocal Separation, Lyrics Transcription, TF-IDF & Analysis

**Project:** AI Music Technical Quality Detector  
**Author:** Ankit Sharda · MSc AI & Data Science · University of Hull

---

## What this notebook does

This notebook handles the lyrics branch of the pipeline. Whether AI-generated songs have weaker lyrical content is an empirical question. The features are extracted and the classifier decides whether they are useful.

1. **Separates vocals** from the full mix using **Demucs** (
2. Transcribes the clean vocal track using **OpenAI Whisper** 
3. Computes hand-crafted lyric features  repetition ratio, vocabulary richness (TTR), structural measures
4. Runs a **Mann-Whitney U test** to check whether AI vs Human differences are statistically significant
5. Runs **TF-IDF** vectorisation to find the most discriminative words per source (AI vs Human)
6. Produces **word clouds** comparing AI-generated vs human lyric vocabulary
7. and more



## 0. Install dependencies



In [ ]:
%pip install -q openai-whisper demucs scikit-learn wordcloud pandas numpy matplotlib seaborn tqdm scipy

## 1. Imports and configuration

Two additions compared to Notebook 1 are: `scipy.stats` for the Mann-Whitney U test, and `demucs` for vocal separation. A `VOCALS_DIR` is defined where separated vocal WAVs are saved; Whisper reads from there instead of the raw MP3.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
import string
import subprocess
import sys
import shutil
import unicodedata

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import whisper

from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm


# Chinese tokenisation
try:
    import jieba
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "jieba",
    ])
    import jieba


# Optional visualisation package
try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False
    print(
        "wordcloud not installed — VIS 5 will be skipped. "
        "Run: pip install wordcloud"
    )


# Paths
AUDIO_FEATURES_CSV = Path("outputs/audio_features.csv")

OUTPUT_DIR = Path("outputs")
TRANSCRIPTS_DIR = OUTPUT_DIR / "transcripts"
VOCALS_DIR = OUTPUT_DIR / "vocals"

OUTPUT_CSV = OUTPUT_DIR / "lyrics_features.csv"
BACKUP_CSV = OUTPUT_DIR / "lyrics_features_before_fix.csv"
TFIDF_CSV = OUTPUT_DIR / "tfidf_top_terms.csv"

OUTPUT_DIR.mkdir(exist_ok=True)
TRANSCRIPTS_DIR.mkdir(exist_ok=True)
VOCALS_DIR.mkdir(exist_ok=True)


# Settings
# large-v3 = highest transcription quality but slower
# small = faster and lighter, but normally less accurate
WHISPER_MODEL_SIZE = "large-v3"

DEMUCS_MODEL = "htdemucs"
USE_DEMUCS = True

# Skip songs already present when transcription is run
RESUME = True

LANG_HINT = {
    "Hindi": "hi",
    "Chinese": "zh",
    "English": "en",
}


print("Configuration ready.")
print(
    "Demucs model  :",
    DEMUCS_MODEL,
    f"(USE_DEMUCS={USE_DEMUCS})",
)
print("Whisper model :", WHISPER_MODEL_SIZE)
print("Vocals dir    :", VOCALS_DIR)
print("Output CSV    :", OUTPUT_CSV)
print("Backup CSV    :", BACKUP_CSV)
print("Jieba version :", jieba.__version__)

## 2. Load manifest

The preferred input is `audio_features.csv` from Notebook 1 (already has validated paths + ok flag). If it doesn't exist yet (Notebook 1 still running), the fallback is to load the raw manifests directly same song list, just without the ok filter.

In [ ]:
if AUDIO_FEATURES_CSV.exists():
    # preferred path  Notebook 1 already ran
    audio_df = pd.read_csv(AUDIO_FEATURES_CSV, low_memory=False)
    manifest = audio_df[audio_df['ok'] == True].copy()
    print('Loaded from audio_features.csv — %d songs' % len(manifest))

else:
    # fallback  build manifest directly from raw CSVs (same as Notebook 1 Section 2)
    print('audio_features.csv not found — loading directly from manifests (Notebook 1 fallback)')
    BASE         = Path('dataset_collection_clean')
    MANIFEST_DIR = BASE / 'manifests'
    MANIFEST_MAP = {
        str(MANIFEST_DIR / 'english_human_final_with_audio.csv'): ('English', 'Human', 'audio_path'),
        str(MANIFEST_DIR / 'hindi_human_final_with_audio.csv')  : ('Hindi',   'Human', 'audio_path'),
        str(MANIFEST_DIR / 'chinese_human_final_with_audio.csv'): ('Chinese', 'Human', 'audio_path'),
        str(BASE / 'ai' / 'english' / 'english_ai_manifest.csv'): ('English', 'AI', 'local_audio_path'),
        str(BASE / 'ai' / 'chinese' / 'chinese_ai_manifest.csv'): ('Chinese', 'AI', 'local_audio_path'),
        str(BASE / 'ai' / 'hindi'   / 'hindi_ai_300_suno_prompts.csv'): ('Hindi', 'AI', 'local_audio_path'),
    }
    frames = []
    for csv_path, (language, source, audio_col) in MANIFEST_MAP.items():
        p = Path(csv_path)
        if not p.exists():
            print('  NOT FOUND (skipping):', csv_path)
            continue
        df = pd.read_csv(p, low_memory=False)
        df['language'] = language
        df['source']   = source
        # normalise audio path column name to 'audio_path'
        if audio_col in df.columns:
            df['audio_path'] = df[audio_col]
        elif 'audio_path' not in df.columns:
            df['audio_path'] = None
        # normalise song_id
        if 'song_id' not in df.columns:
            if 'new_song_id' in df.columns:
                df['song_id'] = df['new_song_id']
            else:
                df['song_id'] = ['%s_%04d' % (source[:2].upper(), i) for i in range(len(df))]
        for col in ['title', 'artist', 'final_genre']:
            if col not in df.columns:
                df[col] = ''
        frames.append(df[['song_id','title','artist','language','final_genre','source','audio_path']])
        print('  Loaded %s: %d rows' % (p.name, len(df)))
    manifest = pd.concat(frames, ignore_index=True)
    # keep only rows where audio file actually exists on disk
    manifest = manifest[manifest['audio_path'].apply(
        lambda x: bool(x) and Path(str(x)).exists()
    )].copy()

print('Songs available: %d' % len(manifest))
print(manifest.groupby(['language', 'source']).size().to_string())

## 3. Load Whisper model

Downloads model weights on first run (~140 MB for `base`). Cached on disk afterwards so subsequent runs are instant. But we used Largev3

In [ ]:
print('Loading Whisper "%s"...' % WHISPER_MODEL_SIZE)
wmodel = whisper.load_model(WHISPER_MODEL_SIZE)
print('Whisper loaded.')

## 3b. Vocal separation Demucs

Demucs splits each song into 4 stems: `vocals`, `drums`, `bass`, `other`. We only keep the `vocals` stem and save it as a WAV file. Whisper then reads this clean vocal track instead of the full mix.



In [ ]:
def separate_vocals(audio_path, song_id):
    """
    Run Demucs on a single audio file and return the path to the vocals WAV.
    Output is cached if the vocals WAV already exists, return it immediately.

    Demucs writes stems to:  <out_dir>/<model>/<song_stem>/vocals.wav
    We copy/move just the vocals.wav to VOCALS_DIR/<song_id>.wav for easy access.

    Returns:
        Path to vocals WAV on success, or None on failure.
    """
    out_wav = VOCALS_DIR / (song_id + '.wav')
    if out_wav.exists():
        return out_wav  # already separated — skip

    # Demucs writes into a temp subdir; we use OUTPUT_DIR / 'demucs_tmp' to keep things tidy
    tmp_dir = OUTPUT_DIR / 'demucs_tmp'
    tmp_dir.mkdir(exist_ok=True)

    try:
        subprocess.run(
            [
                'python', '-m', 'demucs',
                '-n', DEMUCS_MODEL,
                '--two-stems', 'vocals',   # only produce vocals + no-vocals (faster than 4-stem)
                '-o', str(tmp_dir),
                str(audio_path),
            ],
            capture_output=True, timeout=600, check=True
        )
        # Demucs output path: tmp_dir / DEMUCS_MODEL / <audio_stem> / vocals.wav
        audio_stem = Path(audio_path).stem
        vocals_src = tmp_dir / DEMUCS_MODEL / audio_stem / 'vocals.wav'
        if vocals_src.exists():
            vocals_src.rename(out_wav)  # move to our tidy output location
            return out_wav
        else:
            print('  Demucs ran but vocals.wav not found for: %s' % audio_stem)
            return None
    except subprocess.TimeoutExpired:
        print('  Demucs TIMEOUT: %s' % Path(audio_path).name)
        return None
    except subprocess.CalledProcessError as e:
        print('  Demucs FAILED: %s — %s' % (Path(audio_path).name, e.stderr[-200:] if e.stderr else ''))
        return None


# Pre-separation loop 
# We run Demucs over all songs BEFORE the Whisper loop so that:
# (a) we can monitor progress separately, (b) failures are logged up-front
if USE_DEMUCS:
    # use already-loaded manifest (works whether loaded from audio_features.csv or raw manifests)
    _to_separate = manifest[~manifest['song_id'].astype(str).apply(
        lambda sid: (VOCALS_DIR / (sid + '.wav')).exists()
    )]
    already_done = len(manifest) - len(_to_separate)
    print('Demucs: %d songs already separated, %d to process' % (already_done, len(_to_separate)))

    sep_ok, sep_fail = 0, 0
    for _, row in tqdm(_to_separate.iterrows(), total=len(_to_separate), desc='Demucs separation'):
        result = separate_vocals(row['audio_path'], str(row['song_id']))
        if result:
            sep_ok += 1
        else:
            sep_fail += 1

    print('Separation complete — OK: %d | Failed: %d' % (sep_ok, sep_fail))
    print('Vocals saved to:', VOCALS_DIR)
else:
    print('USE_DEMUCS=False — skipping vocal separation. Whisper will use raw MP3s.')

## 4. Feature extraction functions

### 4.1 Transcription

We pass a language hint to Whisper so it doesn't waste time on language detection. If transcription fails for any reason (bad audio, purely instrumental track, etc.) we catch the exception and mark the row as failed rather than crashing the whole loop.

In [ ]:
# Whisper transcription helper.
# NOTE: this cell was empty in the saved notebook. The function below is a
# reconstruction based on how its result is used in the transcription loop
# (keys: ok, error, text, language, detected_language, avg_logprob,
# no_speech_ratio, segments). If you still have the original code, use that.

def transcribe(audio_path, lang_hint=None, no_speech_threshold=0.6):
    try:
        # 1) Language identification on the first 30 s (reported only;
        #    it does not change the transcript)
        audio = whisper.load_audio(str(audio_path))
        clip = whisper.pad_or_trim(audio)
        mel = whisper.log_mel_spectrogram(clip, n_mels=wmodel.dims.n_mels).to(wmodel.device)
        _, probs = wmodel.detect_language(mel)
        detected = max(probs, key=probs.get)

        # 2) Transcription with the expected-language hint
        result = wmodel.transcribe(
            str(audio_path),
            language=lang_hint,
            task='transcribe',
            fp16=torch.cuda.is_available(),
            verbose=False,
        )
        segments = result.get('segments', [])
        logprobs = [s['avg_logprob'] for s in segments if 'avg_logprob' in s]
        no_speech = [s['no_speech_prob'] > no_speech_threshold for s in segments if 'no_speech_prob' in s]
        return {
            'ok': True,
            'error': '',
            'text': '\n'.join(s['text'].strip() for s in segments if s.get('text', '').strip()),
            'language': result.get('language', lang_hint),
            'detected_language': detected,
            'avg_logprob': float(np.mean(logprobs)) if logprobs else None,
            'no_speech_ratio': float(np.mean(no_speech)) if no_speech else None,
            'segments': segments,
        }
    except Exception as exc:
        return {'ok': False, 'error': str(exc)[:300], 'text': '', 'segments': []}


In [ ]:
SCRIPT_PATTERNS = {
    'English': re.compile(r'[A-Za-z]'),
    'Hindi':   re.compile(r'[\u0900-\u097F]'),
    'Chinese': re.compile(r'[\u4E00-\u9FFF\u3400-\u4DBF]'),
}


def script_ratio(text, language):
    pat = SCRIPT_PATTERNS.get(language)
    if pat is None:
        return 1.0
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    return sum(1 for c in letters if pat.match(c)) / len(letters)


# Discard lines where fewer than 50% of alphabetic characters belong to
# the expected script. This removes lines that are entirely platform credits,
# track metadata or foreign-language fragments embedded by Whisper.
def filter_lines(lines, language, min_script=0.5, min_chars=2):
    out = []
    for l in lines:
        l = l.strip()
        if len(l) < min_chars or not any(c.isalpha() for c in l):
            continue
        if script_ratio(l, language) < min_script:
            continue
        out.append(l)
    return out


def filter_tokens(tokens, language):
    pat = SCRIPT_PATTERNS.get(language)
    if pat is None:
        return tokens
    return [t for t in tokens if pat.search(t)]


def _longest_repeated_ngram(tokens, min_len=3, max_len=120):
    n = len(tokens)
    if n < min_len * 2:
        return 0

    def has_repeat(L):
        seen = set()
        for i in range(n - L + 1):
            g = tuple(tokens[i:i + L])
            if g in seen:
                return True
            seen.add(g)
        return False

    lo, hi, best = min_len, min(max_len, n // 2), 0  # binary search: find longest repeated n-gram
    while lo <= hi:
        mid = (lo + hi) // 2
        if has_repeat(mid):  # if length mid repeats, try longer
            best, lo = mid, mid + 1  # record best and search longer half
        else:
            hi = mid - 1  # no repeat at this length, search shorter half
    return best


def longest_repeated_segment(text, language):
    if not text:
        return 0, 0.0
    lines = filter_lines([l for l in str(text).split('\n') if l.strip()], language)
    if not lines:
        return 0, 0.0
    tokens = filter_tokens(tokenize_text(' '.join(lines), language), language)
    if not tokens:
        return 0, 0.0
    best = _longest_repeated_ngram(tokens)
    return best, round(best / len(tokens), 4)

In [ ]:
def norm_line(line):
    line = line.casefold().translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', line).strip()


def repetition_features(lines, language):
    norm   = [norm_line(l) for l in lines if norm_line(l)]
    tokens = [w for line in norm for w in tokenize_text(line, language)]
    # filter_lines guards against off-script/short fragments.
    # Re-join tokens to a string so filter_lines can apply its script-ratio
    # check, then split back into tokens via tokenize_text.
    tokens = [w for line in filter_lines([' '.join(norm)], language)
              for w in tokenize_text(line, language)]
    if not tokens:
        return {'repetition_ratio': None, 'bigram_rep_ratio': None}

    counts   = Counter(tokens)
    repeated = sum(v - 1 for v in counts.values() if v > 1)  # extra occurrences beyond the first
    bigrams  = list(zip(tokens, tokens[1:]))
    bg_rep   = sum(v - 1 for v in Counter(bigrams).values() if v > 1)

    return {
        'repetition_ratio': round(repeated / len(tokens), 4),
        'bigram_rep_ratio': round(bg_rep / len(bigrams), 4) if bigrams else None,
    }

### 4.2 Repetition features

$$\text{Repetition Ratio} = \frac{\text{repeated tokens}}{\text{total tokens}}$$

Tokens are produced by `tokenize_text()`, which is language-aware: regex word
tokenisation for English, Devanagari-aware tokenisation for Hindi, and jieba
word segmentation for Mandarin. Whitespace splitting is not used, because
Mandarin transcripts contain no inter-word spaces and would otherwise yield
near-zero token counts.



In [ ]:
# Language-aware tokeniser: English uses word regex, Hindi uses Devanagari
# character ranges, and Mandarin uses jieba because Chinese text has no
# spaces between words whitespace splitting would give near-zero token counts.
def tokenize_text(text, language):
    text = str(text).casefold()

    if language == 'Chinese':
        return [w.strip() for w in jieba.lcut(text) if w.strip() and re.search(r'[\u4e00-\u9fffA-Za-z0-9]', w)]  # drop punctuation-only tokens

    if language == 'Hindi':
        return re.findall(r'[\u0900-\u097F]+|[A-Za-z]+', text)  # Devanagari block + Latin (romanised Hindi)

    return re.findall(r"[A-Za-z]+(?:['’][A-Za-z]+)?", text)  # keeps contractions like "don't"
    
def norm_line(line):
    line = line.casefold().translate(str.maketrans('','',string.punctuation))
    return re.sub(r'\s+', ' ', line).strip()


### 4.3 Vocabulary features

**Type-Token Ratio (TTR):**

$$\text{TTR} = \frac{\text{unique words}}{\text{total words}}$$

A low TTR (near 0) means the same small set of words repeats constantly weak vocabulary. A high TTR means the song uses many different words, which tends to be more characteristic of human writing. Note that TTR is sensitive to text length, so we also record `total_words` so we can account for that later.

In [ ]:
# TTR (type-token ratio) = unique words / total words.
# A low TTR means many words repeat; a high TTR means richer vocabulary.
# TTR shrinks naturally as transcripts get longer, so raw counts
# should not be compared across transcripts of very different lengths.
def vocab_features(text, language):
    tokens = tokenize_text(text, language)
    if not tokens:
        return {'ttr': None, 'vocab_size': 0, 'total_words': 0,
                'avg_word_length': None, 'rare_word_count': 0}
    freq = Counter(tokens)
    return {
        'total_words'    : len(tokens),
        'vocab_size'     : len(freq),
        'ttr'            : round(len(freq) / len(tokens), 4),
        'avg_word_length': round(float(np.mean([len(w) for w in tokens])), 3),
        'rare_word_count': sum(1 for v in freq.values() if v == 1),  # hapax legomena: words appearing exactly once
    }


def structure_features(text, language):
    lines     = text.split('\n') if text else []
    non_empty = [l for l in lines if l.strip()]
    wc        = [len(tokenize_text(l, language)) for l in non_empty]
    segs, cur = [], []  # segs = list of verse/chorus blocks; cur = current block being built
    for l in lines:
        if l.strip(): cur.append(l)
        elif cur:     segs.append(cur); cur = []  # blank line ends a segment
    if cur: segs.append(cur)
    return {
        'total_lines'       : len(lines),
        'non_empty_lines'   : len(non_empty),
        'avg_words_per_line': round(float(np.mean(wc)), 3) if wc else None,
        'max_words_per_line': max(wc) if wc else None,
        'segment_count'     : len(segs),
        'empty_line_ratio'  : round(sum(not l.strip() for l in lines) / len(lines), 4) if lines else None,
    }

## 5. Main transcription loop

In this Whisper processes each audio file one at a time. On CPU it takes several minutes per song. We save progress incrementally (same pattern as Notebook 1) so it's safe to interrupt and resume.

**Input priority:** If `USE_DEMUCS=True` and a vocals WAV exists for the song, Whisper receives the clean vocal track. If separation failed for a song, we fall back to the original MP3 so we don't lose that song entirely.



In [ ]:
done, existing = set(), pd.DataFrame()
if RESUME and OUTPUT_CSV.exists():
    existing = pd.read_csv(OUTPUT_CSV, low_memory=False)
    done     = set(existing['song_id'].astype(str))
    print('Resuming: %d already done.' % len(done))

# Each song is passed with a language hint so Whisper does not spend
# time on automatic language detection, which reduces errors on
# non-English audio with mixed-language credits or instrumental sections.
to_do = manifest[~manifest['song_id'].astype(str).isin(done)]
print('To transcribe: %d songs' % len(to_do))

records = []
for _, row in tqdm(to_do.iterrows(), total=len(to_do), desc='Transcribing'):
    sid  = str(row.get('song_id',''))
    lang = str(row.get('language',''))
    rec  = {
        'song_id'                    : sid,
        'title'                      : row.get('title',''),
        'artist'                     : row.get('artist',''),
        'language'                   : lang,
        'final_genre'                : row.get('final_genre',''),
        'source'                     : row.get('source',''),
        'transcript_ok'              : False,
        'transcript_error'           : '',
        'whisper_lang'               : '',
        'whisper_detected_lang'      : '',
        'whisper_lang_match'         : None,
        'whisper_avg_logprob'        : None,
        'whisper_no_speech_ratio'    : None,
    }

    vocals_wav = VOCALS_DIR / (sid + '.wav')
    input_path = str(vocals_wav) if (USE_DEMUCS and vocals_wav.exists()) else row['audio_path']
    rec['demucs_used'] = USE_DEMUCS and vocals_wav.exists()

    t = transcribe(input_path, lang_hint=LANG_HINT.get(lang))
    if not t['ok']:
        rec['transcript_error'] = t['error']
    else:
        text         = t['text']
        expected_iso = LANG_HINT.get(lang, '').lower()
        detected_iso = str(t.get('detected_language','')).lower()

        rec['whisper_lang']              = t['language']
        rec['whisper_detected_lang']     = detected_iso
        rec['whisper_lang_match']        = int(detected_iso == expected_iso) if detected_iso else None  # 1=correct language, 0=wrong
        rec['whisper_avg_logprob']       = t.get('avg_logprob')
        rec['whisper_no_speech_ratio']   = t.get('no_speech_ratio')
        rec['transcript_ok']             = True

        (TRANSCRIPTS_DIR / (sid + '.txt')).write_text(text, encoding='utf-8')

        lines = [s['text'].strip() for s in t['segments'] if s.get('text','').strip()]
        if not lines and text:
            lines = [l.strip() for l in text.split('\n') if l.strip()]

        rec.update(structure_features(text, lang))
        rec.update(vocab_features(text, lang))
        rec.update(repetition_features(lines, lang))

        repeat_words, repeat_ratio = longest_repeated_segment(text, lang)
        rec['longest_repeat_words'] = repeat_words
        rec['longest_repeat_ratio'] = repeat_ratio

    records.append(rec)

    batch   = pd.DataFrame(records)
    out     = pd.concat([existing, batch], ignore_index=True) if not existing.empty else batch
    tmp_csv = OUTPUT_CSV.with_suffix('.tmp')
    out.to_csv(tmp_csv, index=False, encoding='utf-8-sig')
    tmp_csv.replace(OUTPUT_CSV)

print('Done. Successful: %d / %d' % (sum(r['transcript_ok'] for r in records), len(records)))

## cleaning unwanted Mandarin data from the transcripts like intro and other ads

In [ ]:

# Recompute lyric features from existing transcripts
df = pd.read_csv(OUTPUT_CSV, low_memory=False)

backup_csv = OUTPUT_CSV.with_name('lyrics_features_before_recompute.csv')
if not backup_csv.exists():
    shutil.copy2(OUTPUT_CSV, backup_csv)
    print('Backup saved:', backup_csv)

patched, missing = 0, 0

for i, row in tqdm(df.iterrows(), total=len(df), desc='Updating lyric features'):
    if str(row.get('transcript_ok','')).lower() != 'true':
        continue

    sid, lang = str(row['song_id']), str(row['language'])
    txt       = TRANSCRIPTS_DIR / f'{sid}.txt'
    backup    = TRANSCRIPTS_DIR / f'{sid}.txt.backup'
    src       = backup if backup.exists() else txt

    if not src.exists():
        missing += 1
        continue

    text  = src.read_text(encoding='utf-8', errors='ignore').strip()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines and text:
        lines = [text]

    feats = {}
    feats.update(structure_features(text, lang))
    feats.update(vocab_features(text, lang))
    feats.update(repetition_features(lines, lang))

    repeat_words, repeat_ratio = longest_repeated_segment(text, lang)
    feats['longest_repeat_words'] = repeat_words
    feats['longest_repeat_ratio'] = repeat_ratio

    for feature, value in feats.items():
        df.at[i, feature] = value

    df.at[i, 'full_text'] = text
    patched += 1

df.drop(
    columns=['lyric_perplexity', 'unique_line_ratio'],
    errors='ignore',
    inplace=True
)

df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print('Updated:', patched, 'rows')
print('Missing transcript files:', missing)
print('Saved:', OUTPUT_CSV)

### Recompute lyric features from existing transcripts          


In [ ]:
# Tokenisation audit: whitespace splitting vs jieba on Mandarin
# Recomputes the pre-fix state from the same transcripts.
# Self-contained: does not depend on earlier cells.


import re, string
import pandas as pd
from pathlib import Path
from collections import Counter

OUT_DIR    = Path('outputs')
TRANS_DIR  = OUT_DIR / 'transcripts'
FEATS_CSV  = OUT_DIR / 'lyrics_features.csv'

def _norm(line):
    line = line.casefold().translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', line).strip()

feats = pd.read_csv(FEATS_CSV, low_memory=False)
zh = feats[(feats.language == 'Chinese') &
           (feats.transcript_ok.astype(str).str.lower() == 'true')]
print(f'Mandarin tracks with transcripts: {len(zh)}')

rows = []
for _, r in zh.iterrows():
    sid = str(r.song_id)
    bak, txt = TRANS_DIR / f'{sid}.txt.backup', TRANS_DIR / f'{sid}.txt'
    p = bak if bak.exists() else txt
    if not p.exists():
        continue
    text = p.read_text(encoding='utf-8', errors='ignore')

    ws = [w for line in text.split('\n') for w in _norm(line).split()]
    c = Counter(ws)
    rep_ws = (sum(v - 1 for v in c.values() if v > 1) / len(ws)) if ws else 0.0

    letters = [ch for ch in text if ch.isalpha()]
    latin = sum(1 for ch in letters if re.match(r'[A-Za-z]', ch)) / len(letters) if letters else 0.0

    rows.append({'source': r.source, 'n_ws': len(ws), 'rep_ws': rep_ws, 'latin': latin})

aud = pd.DataFrame(rows)
print(f'Transcripts read: {len(aud)}\n')

print('--- Whitespace splitting (the original method) ---')
for src in ['AI', 'Human']:
    g = aud[aud.source == src]
    print(f'  {src:<5} median {g.n_ws.median():.1f} tokens, '
          f'{(g.rep_ws == 0).mean():.1%} zero-repetition')

h = aud[aud.source == 'Human']
print(f'\n  Human non-zero cases: median Latin fraction '
      f'{h.loc[h.rep_ws > 0, "latin"].median():.3f}')
print(f'  Human zero cases    : median Latin fraction '
      f'{h.loc[h.rep_ws == 0, "latin"].median():.3f}')

print('\n--- After jieba segmentation ---')
for src in ['AI', 'Human']:
    g = zh[zh.source == src]
    print(f'  {src:<5} {(pd.to_numeric(g.repetition_ratio, errors="coerce") == 0).mean():.1%} '
          f'zero-repetition')

## 6. Summary table

Descriptive statistics for the lyric features split by source. This is the first look at whether AI lyrics are measurably different from human lyrics. Larger differences in means would indicate stronger separability for the classifier.

In [ ]:
# Canonical successful dataset + lyric feature summary
df = pd.read_csv(OUTPUT_CSV, low_memory=False)

ok = df['transcript_ok'].astype(str).str.lower() == 'true'
df_ok = (
    df.loc[ok]
      .drop_duplicates(subset='song_id', keep='last')
      .reset_index(drop=True)
)

lyr = df_ok.copy()
_lyr_path = OUTPUT_CSV

print('CSV rows              :', len(df))
print('Successful unique     :', len(df_ok))
print('Duplicate song IDs    :', df_ok['song_id'].duplicated().sum())
print()
print('Tracks by language and source:')
display(df_ok.groupby(['language','source']).size().unstack(fill_value=0))

LYRIC_KF = [
    'repetition_ratio', 'bigram_rep_ratio', 'longest_repeat_ratio',
    'ttr', 'vocab_size', 'total_words', 'avg_word_length', 'rare_word_count',
]

kf = [
    c for c in LYRIC_KF
    if c in df_ok.columns and df_ok[c].nunique(dropna=True) > 1
]

for c in kf:
    df_ok[c] = pd.to_numeric(df_ok[c], errors='coerce')

summary = df_ok.groupby('source')[kf].agg(['mean','std']).round(4)
summary.columns = [' '.join(c) for c in summary.columns]

print('=== Lyric Feature Summary: AI vs Human ===')
display(summary.T)

## VIS 4 Repetition ratio and TTR distributions

Histograms showing how the two groups are distributed on our two most important lyric features. Separation between the AI (red) and Human (blue) peaks would suggest these features carry useful signal for the classifier.

In [ ]:
# VIS 4 Lyric feature distributions
palette = {'AI': '#e05252', 'Human': '#4a90d9'}
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (col, xlabel, title) in zip(axes, [
    ('repetition_ratio', 'Repeated-token ratio', 'Token Repetition Ratio'),
    ('ttr', 'Type-token ratio (length-sensitive)', 'Type-Token Ratio (TTR)'),
]):
    if col not in df_ok.columns: continue
    for src, grp in df_ok.groupby('source'):
        vals = pd.to_numeric(grp[col], errors='coerce').dropna()
        ax.hist(vals, bins=20, alpha=0.65, color=palette.get(src, 'grey'),
                edgecolor='white', label=src)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

fig.suptitle('Lyric Feature Distributions: AI vs Human',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'vis4_lyric_distributions.png',
            bbox_inches='tight', dpi=150)
plt.show()

print('Saved: vis4_lyric_distributions.png')

## 6b. Statistical significance Mann-Whitney U test

The summary table shows means but does not indicate whether the differences are statistically reliable. The Mann-Whitney U test is a non-parametric test it doesn't assume the data is normally distributed, which is important here because lyric features are often skewed.

**Null hypothesis:** AI and Human lyrics come from the same distribution for a given feature.

A p-value below 0.05 provides grounds to reject the null hypothesis.

In [ ]:
# Pooled Mann-Whitney U Tests: AI vs Human
# Secondary analysis across all languages
# Benjamini-Hochberg FDR correction adjusts p-values so the expected fraction
# of false discoveries stays at 5%, controlling for multiple comparisons.
from statsmodels.stats.multitest import multipletests

TEST_FEATURES = [
    'repetition_ratio', 'bigram_rep_ratio', 'longest_repeat_ratio',
    'ttr', 'vocab_size', 'total_words', 'avg_word_length', 'rare_word_count',
]

sig_rows = []

for feat in TEST_FEATURES:
    if feat not in df_ok.columns:
        continue

    ai = pd.to_numeric(
        df_ok.loc[df_ok['source'] == 'AI', feat], errors='coerce'
    ).dropna()

    hu = pd.to_numeric(
        df_ok.loc[df_ok['source'] == 'Human', feat], errors='coerce'
    ).dropna()

    if len(ai) < 5 or len(hu) < 5:
        continue

    if pd.concat([ai, hu]).nunique() <= 1:
        continue

    u, p = stats.mannwhitneyu(ai, hu, alternative='two-sided')
    effect = (2 * u / (len(ai) * len(hu))) - 1  # rank-biserial: +1=AI always higher, -1=Human always higher

    sig_rows.append({
        'feature'       : feat,
        'AI_mean'       : ai.mean(),
        'Human_mean'    : hu.mean(),
        'rank_biserial' : effect,
        'U_statistic'   : u,
        'p_value'       : p,
        'n_AI'          : len(ai),
        'n_Human'       : len(hu),
    })

sig_df = pd.DataFrame(sig_rows)

if not sig_df.empty:
    sig_df['p_FDR'] = multipletests(
        sig_df['p_value'], method='fdr_bh'
    )[1]
    sig_df['significant_FDR'] = sig_df['p_FDR'] < 0.05

print('=== Pooled Mann-Whitney U Test: AI vs Human ===')
print('FDR correction applied across all tested features')
print('Note: within-language tests are the primary analysis.')

display(sig_df.style.format({
    'AI_mean'       : '{:.4f}',
    'Human_mean'    : '{:.4f}',
    'rank_biserial' : '{:.4f}',
    'U_statistic'   : '{:.1f}',
    'p_value'       : '{:.3e}',
    'p_FDR'         : '{:.3e}',
}))

## 6c. Statistical Significance of Whisper ASR Diagnostics



The analysis investigates whether AI and human recordings produce different
Whisper ASR behaviour. These measures are not treated as direct measurements
of pronunciation quality, accent naturalness or native phonetic accuracy.
Observed differences may also reflect recording source, vocal separation,
instrumental interference, singing style and arrangement.


In [ ]:
# Whisper ASR Diagnostics by Language
# These are not direct pronunciation or accent measurements
from statsmodels.stats.multitest import multipletests

ASR_FEATS = [
    'whisper_avg_logprob',
    'whisper_no_speech_ratio',
    'whisper_lang_match',
]

rows = []

for lang in ['English', 'Hindi', 'Chinese']:
    part = df_ok[df_ok['language'] == lang]

    for feat in ASR_FEATS:
        if feat not in part.columns:
            continue

        ai = pd.to_numeric(
            part.loc[part['source'] == 'AI', feat],
            errors='coerce'
        ).dropna()

        hu = pd.to_numeric(
            part.loc[part['source'] == 'Human', feat],
            errors='coerce'
        ).dropna()

        if len(ai) < 5 or len(hu) < 5:
            continue

        u, p = stats.mannwhitneyu(
            ai, hu, alternative='two-sided'
        )

        effect = (2 * u / (len(ai) * len(hu))) - 1

        rows.append({
            'language'      : lang,
            'feature'       : feat,
            'AI_mean'       : ai.mean(),
            'Human_mean'    : hu.mean(),
            'rank_biserial' : effect,
            'p_value'       : p,
            'n_AI'          : len(ai),
            'n_Human'       : len(hu),
        })

asr_tests = pd.DataFrame(rows)

if not asr_tests.empty:
    asr_tests['p_FDR'] = multipletests(
        asr_tests['p_value'],
        method='fdr_bh'
    )[1]

    asr_tests['significant_FDR'] = (
        asr_tests['p_FDR'] < 0.05
    )

print('=== Whisper ASR Diagnostics by Language ===')
print('These measures describe ASR behaviour, not pronunciation quality.')

display(asr_tests.style.format({
    'AI_mean'       : '{:.4f}',
    'Human_mean'    : '{:.4f}',
    'rank_biserial' : '{:.4f}',
    'p_value'       : '{:.3e}',
    'p_FDR'         : '{:.3e}',
}))

save_path = OUTPUT_DIR / 'asr_diagnostics_by_language.csv'
asr_tests.to_csv(
    save_path,
    index=False,
    encoding='utf-8-sig'
)

print('Saved:', save_path)

print('\n=== Whisper Detected Languages for Hindi Tracks ===')

hindi = df_ok[df_ok['language'] == 'Hindi']

detected = pd.crosstab(
    hindi['whisper_detected_lang'],
    hindi['source']
)

display(detected.sort_values(
    by=[c for c in ['AI', 'Human'] if c in detected.columns],
    ascending=False
).head(10))

## VIS 4b. Whisper ASR Diagnostic Distributions

The following box plots compare Whisper's average transcription confidence and
no-speech ratio between AI and human tracks.

These measures describe **ASR behaviour only**. Differences must not be
interpreted as direct evidence of pronunciation quality, accent naturalness or
native-language phonetics because they may also reflect recording source,
vocal separation, instrumental interference and singing style.

In [ ]:
# VIS 4b Whisper ASR Diagnostics by Language and Source
asr_plot_cols = [
    'whisper_avg_logprob',
    'whisper_no_speech_ratio',
]

labels = {
    'whisper_avg_logprob'    : 'Whisper Average Log-Probability',
    'whisper_no_speech_ratio': 'Whisper No-Speech Ratio',
}

palette = {'AI': '#e05252', 'Human': '#4a90d9'}
languages = ['English', 'Hindi', 'Chinese']

fig, axes = plt.subplots(3, 2, figsize=(13, 15))

for row, lang in enumerate(languages):
    part = df_ok[df_ok['language'] == lang]

    for col, feat in enumerate(asr_plot_cols):
        ax = axes[row, col]

        plot_data = part[['source', feat]].copy()
        plot_data[feat] = pd.to_numeric(
            plot_data[feat], errors='coerce'
        )
        plot_data = plot_data.dropna()

        sns.boxplot(
            data=plot_data,
            x='source',
            y=feat,
            order=['AI', 'Human'],
            palette=palette,
            width=0.5,
            fliersize=3,
            ax=ax
        )

        display_lang = 'Mandarin Chinese' if lang == 'Chinese' else lang

        ax.set_title(
            f'{display_lang} {labels[feat]}',
            fontsize=11,
            fontweight='bold'
        )
        ax.set_xlabel('')
        ax.set_ylabel(labels[feat])

fig.suptitle(
    'Whisper ASR Diagnostic Distributions by Language',
    fontsize=14,
    fontweight='bold',
    y=1.01
)

plt.tight_layout()

save_path = OUTPUT_DIR / 'vis4b_asr_diagnostics_by_language.png'
plt.savefig(save_path, bbox_inches='tight', dpi=150)
plt.show()

print('Saved:', save_path)

## 7. TF-IDF analysis

**TF-IDF** (Term Frequency–Inverse Document Frequency) weights each word by how often it appears in one group but NOT in the other. This surfaces the most characteristic vocabulary of AI vs Human lyrics.

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\frac{N}{\text{DF}(t)}$$

where $N$ = total documents, $\text{DF}(t)$ = number of documents containing term $t$.

In [ ]:
# Exploratory TF-IDF Terms by Language and Source
def load_transcript(sid):
    txt    = TRANSCRIPTS_DIR / f'{sid}.txt'
    backup = TRANSCRIPTS_DIR / f'{sid}.txt.backup'
    src    = backup if backup.exists() else txt
    return src.read_text(encoding='utf-8', errors='ignore') if src.exists() else ''

STOPWORDS = {
    'English': {
        'the','a','an','and','or','of','in','on','to','for','with','this','that',
        'is','are','was','were','be','been','we','our','i','you','he','she','it',
        'they','my','your','me','us','but','as','at','from','by',
        'thank','thanks','yeah','ooh','oh','wanna','gonna','gotta'
    },
    'Hindi': {
        'है','हैं','था','थी','थे','का','की','के','को','में','से','और','यह','वह',
        'मैं','हम','तुम','आप','मेरे','तेरे','जो','भी','तो','पर','एक','नहीं',
        'प्रश्न','अब','ही','हो','ना','ये','तू','सब','कुछ','आज','कह','कर','चल','बस','फिर'
    },
    'Chinese': {
        '的','了','在','是','我','你','他','她','它','我们','你们','他们','和','也',
        '都','就','不','有','这','那','着','吗','呢','吧','一个',
        '李宗盛','明镜','明鏡','点点','點點','栏目','欄目','打赏','打賞','支持',
        '订阅','訂閱','作词','作詞','作曲','制作','製作','视频','視頻',
        '字幕','翻译','翻譯','请','請','与','與','为','為','被','把','再','来','來'
    },
}

df_ok['full_text'] = df_ok['song_id'].apply(load_transcript)

rows = []

for lang in ['English','Hindi','Chinese']:
    part = df_ok[
        (df_ok['language'] == lang) &
        (df_ok['full_text'].fillna('').str.strip() != '')
    ].copy()

    if part['source'].nunique() < 2:
        continue

    def tfidf_tokenizer(text):
        return [
            token for token in tokenize_text(text, lang)
            if token not in STOPWORDS.get(lang, set()) and token.strip()
        ]

    vectoriser = TfidfVectorizer(
        tokenizer=tfidf_tokenizer,
        token_pattern=None,
        lowercase=False,
        min_df=5,
        max_features=500,
        ngram_range=(1, 2),
        sublinear_tf=True
    )

    try:
        matrix = vectoriser.fit_transform(part['full_text'])
    except ValueError as e:
        print(f'{lang} skipped:', e)
        continue

    terms = vectoriser.get_feature_names_out()
    scores = pd.DataFrame(
        matrix.toarray(),
        columns=terms,
        index=part.index
    )

    for src in ['AI','Human']:
        idx = part.index[part['source'] == src]
        top = scores.loc[idx].mean(axis=0).nlargest(20)

        for term, score in top.items():
            rows.append({
                'language'   : lang,
                'source'     : src,
                'term'       : term,
                'tfidf_score': float(score)
            })

tfidf_top = pd.DataFrame(rows)
tfidf_top.to_csv(TFIDF_CSV, index=False, encoding='utf-8-sig')

print('TF-IDF top terms saved to:', TFIDF_CSV)
display(tfidf_top.groupby(['language','source']).size().unstack(fill_value=0))
display(tfidf_top.groupby(['language','source']).head(10).round(5))

## VIS 5 Word clouds: AI lyrics vs Human lyrics

Word clouds give an intuitive visual picture of the vocabulary each source uses. Larger words appear more often. If AI lyrics are more repetitive, the AI cloud would show a smaller, more repeated vocabulary.

In [ ]:
# VIS 5: Exploratory Lyric Word Clouds by Language and Source
from wordcloud import WordCloud
from matplotlib import font_manager
from collections import Counter

def find_font(names):
    for name in names:
        try:
            path = font_manager.findfont(name, fallback_to_default=False)
            if path:
                return path
        except Exception:
            continue
    return None

FONTS = {
    'English': None,
    'Hindi'  : find_font(['Nirmala UI', 'Mangal', 'Noto Sans Devanagari']),
    'Chinese': find_font(['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC']),
}

def cloud_freq(group, lang):
    words = []
    for text in group['full_text'].fillna(''):
        words.extend([
            word for word in tokenize_text(text, lang)
            if word not in STOPWORDS.get(lang, set())
            and len(word.strip()) > 1
        ])
    return Counter(words)

fig, axes = plt.subplots(3, 2, figsize=(16, 17))

for row, lang in enumerate(['English', 'Hindi', 'Chinese']):
    lang_data = df_ok[df_ok['language'] == lang]

    for col, source in enumerate(['AI', 'Human']):
        ax = axes[row, col]
        group = lang_data[lang_data['source'] == source]
        frequencies = cloud_freq(group, lang)

        if not frequencies:
            ax.text(0.5, 0.5, 'No lyric text available',
                    ha='center', va='center')
            ax.axis('off')
            continue

        if lang != 'English' and FONTS.get(lang) is None:
            ax.text(0.5, 0.5, f'{lang} font not found',
                    ha='center', va='center')
            ax.axis('off')
            continue

        word_cloud = WordCloud(
            width=700,
            height=400,
            background_color='white',
            colormap='Reds' if source == 'AI' else 'Blues',
            font_path=FONTS.get(lang),
            max_words=100,
            collocations=False
        ).generate_from_frequencies(frequencies)

        ax.imshow(word_cloud, interpolation='bilinear')
        ax.axis('off')

        display_language = 'Mandarin Chinese' if lang == 'Chinese' else lang
        ax.set_title(
            f'{source} {display_language}',
            fontsize=14,
            fontweight='bold'
        )

fig.suptitle(
    'Exploratory Lyric Word Clouds: AI-Generated vs Human Songs',
    fontsize=16,
    fontweight='bold',
    y=1.01
)

plt.tight_layout()

save_path = OUTPUT_DIR / 'vis5_word_clouds_all_languages.png'
plt.savefig(save_path, bbox_inches='tight', dpi=150)
plt.show()

print('Saved:', save_path)
print('Hindi font:', FONTS['Hindi'])
print('Chinese font:', FONTS['Chinese'])

## VIS 6  TF-IDF top-20 discriminative terms per source

This bar chart shows the 20 most characteristic terms for AI-generated songs and the 20 most characteristic for human songs. Terms with high TF-IDF in one group but not the other are the strongest linguistic separators.

In [ ]:
# VIS 6: Top Mean TF-IDF Terms by Language and Source
from matplotlib.font_manager import FontProperties

if 'tfidf_top' in globals() and not tfidf_top.empty:
    palette   = {'AI': '#e05252', 'Human': '#4a90d9'}
    languages = ['English', 'Hindi', 'Chinese']
    sources   = ['AI', 'Human']

    tfidf_top['tfidf_score'] = pd.to_numeric(
        tfidf_top['tfidf_score'], errors='coerce'
    )

    fig, axes = plt.subplots(3, 2, figsize=(17, 18))

    for row, lang in enumerate(languages):
        for col, src in enumerate(sources):
            ax = axes[row, col]

            sub = (
                tfidf_top[
                    (tfidf_top['language'] == lang) &
                    (tfidf_top['source'] == src)
                ]
                .dropna(subset=['tfidf_score'])
                .nlargest(15, 'tfidf_score')
                .sort_values('tfidf_score')
            )

            if sub.empty:
                ax.text(0.5, 0.5, 'No TF-IDF data',
                        ha='center', va='center')
                ax.axis('off')
                continue

            ax.barh(
                sub['term'],
                sub['tfidf_score'],
                color=palette[src],
                edgecolor='white'
            )

            if lang != 'English' and FONTS.get(lang):
                font_prop = FontProperties(
                    fname=FONTS[lang], size=10
                )
                for label in ax.get_yticklabels():
                    label.set_fontproperties(font_prop)

            display_lang = (
                'Mandarin Chinese' if lang == 'Chinese' else lang
            )

            ax.set_title(
                f'{src} {display_lang}',
                fontsize=12,
                fontweight='bold'
            )
            ax.set_xlabel('Mean TF-IDF Score')
            ax.set_ylabel('')
            ax.tick_params(axis='y', labelsize=9)

    fig.suptitle(
        'Top Mean Lyric TF-IDF Terms by Language and Source',
        fontsize=16,
        fontweight='bold',
        y=1.01
    )

    plt.tight_layout()

    save_path = OUTPUT_DIR / 'vis6_tfidf_terms_by_language.png'
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()

    print('Saved:', save_path)
    print('Note: TF-IDF terms are exploratory, not significance tests.')
else:
    print('VIS 6 skipped: TF-IDF results not available.')

## 8. Within-Language Lyric Feature Analysis

This analysis compares AI-generated and human tracks separately within English, Hindi and Mandarin.

The purpose is to determine whether lyric-feature differences are consistent
across languages or whether their direction and magnitude depend on language.
The tested features include token repetition, bigram repetition, longest repeated
span, type-token ratio, vocabulary size, transcript length, average token length
and rare-word count.



In [ ]:
# Within-Language Lyric Features: AI vs Human
# FDR correction across all language-feature comparisons
from statsmodels.stats.multitest import multipletests

CONTENT_FEATS = [
    'repetition_ratio', 'bigram_rep_ratio', 'longest_repeat_ratio',
    'ttr', 'vocab_size', 'total_words', 'avg_word_length', 'rare_word_count',
]

rows = []

for lang in ['English', 'Hindi', 'Chinese']:
    part = df_ok[df_ok['language'] == lang]

    for feat in CONTENT_FEATS:
        if feat not in part.columns:
            continue

        ai = pd.to_numeric(
            part.loc[part['source'] == 'AI', feat],
            errors='coerce'
        ).dropna()

        hu = pd.to_numeric(
            part.loc[part['source'] == 'Human', feat],
            errors='coerce'
        ).dropna()

        if len(ai) < 5 or len(hu) < 5:
            continue

        if pd.concat([ai, hu]).nunique() <= 1:
            continue

        u, p = stats.mannwhitneyu(
            ai, hu, alternative='two-sided'
        )
        effect = (2 * u / (len(ai) * len(hu))) - 1  # rank-biserial correlation

        rows.append({
            'language'   : lang,
            'feature'    : feat,
            'AI_mean'    : ai.mean(),
            'Human_mean' : hu.mean(),
            'effect'     : effect,
            'p_value'    : p,
            'n_AI'       : len(ai),
            'n_Human'    : len(hu),
        })

language_tests = pd.DataFrame(rows)

if not language_tests.empty:
    language_tests['p_FDR'] = multipletests(
        language_tests['p_value'],
        method='fdr_bh'
    )[1]
    language_tests['significant_FDR'] = (
        language_tests['p_FDR'] < 0.05
    )

print('=== Within-Language Lyric Feature Tests ===')
print('FDR correction applied across all language-feature comparisons.')

display(language_tests.style.format({
    'AI_mean'    : '{:.4f}',
    'Human_mean' : '{:.4f}',
    'effect'     : '{:.4f}',
    'p_value'    : '{:.3e}',
    'p_FDR'      : '{:.3e}',
}))

save_path = OUTPUT_DIR / 'lyrics_within_language_tests.csv'
language_tests.to_csv(
    save_path,
    index=False,
    encoding='utf-8-sig'
)

print('Saved:', save_path)

In [ ]:
# Mean and Median Check for Selected Within-Language Features
CHECK_FEATS = ['avg_word_length', 'ttr', 'total_words']

rows = []

for lang in ['English', 'Hindi', 'Chinese']:
    part = df_ok[df_ok['language'] == lang]

    for feat in CHECK_FEATS:
        if feat not in part.columns:
            continue

        for src in ['AI', 'Human']:
            values = pd.to_numeric(
                part.loc[part['source'] == src, feat],
                errors='coerce'
            ).dropna()

            if values.empty:
                continue

            rows.append({
                'language': lang,
                'feature' : feat,
                'source'  : src,
                'mean'    : values.mean(),
                'median'  : values.median(),
                'Q1'      : values.quantile(0.25),
                'Q3'      : values.quantile(0.75),
                'n'       : len(values),
            })

distribution_check = pd.DataFrame(rows)

display(distribution_check.style.format({
    'mean'  : '{:.4f}',
    'median': '{:.4f}',
    'Q1'    : '{:.4f}',
    'Q3'    : '{:.4f}',
}))

save_path = OUTPUT_DIR / 'lyrics_distribution_check.csv'
distribution_check.to_csv(
    save_path,
    index=False,
    encoding='utf-8-sig'
)

print('Saved:', save_path)

In [ ]:
# Language-Specific Lyric Feature Effect Sizes
# Positive = higher in AI, Negative = higher in Human
if 'language_tests' in globals() and not language_tests.empty:
    feature_labels = {
        'repetition_ratio'    : 'Token repetition',
        'bigram_rep_ratio'    : 'Bigram repetition',
        'longest_repeat_ratio': 'Longest repeat ratio',
        'ttr'                 : 'Type-token ratio',
        'vocab_size'          : 'Vocabulary size',
        'total_words'         : 'Transcript length',
        'avg_word_length'     : 'Average token length',
        'rare_word_count'     : 'Rare-word count',
    }

    plot_df = language_tests.copy()
    plot_df['feature_label'] = plot_df['feature'].map(feature_labels)

    heat = plot_df.pivot(
        index='feature_label',
        columns='language',
        values='effect'
    )[['English', 'Hindi', 'Chinese']]

    sig = plot_df.pivot(
        index='feature_label',
        columns='language',
        values='significant_FDR'
    )[['English', 'Hindi', 'Chinese']]

    annot = heat.copy().astype(str)
    for feat in heat.index:
        for lang in heat.columns:
            val = heat.loc[feat, lang]
            star = '*' if sig.loc[feat, lang] else ''
            annot.loc[feat, lang] = f'{val:.2f}{star}'

    plt.figure(figsize=(9, 6))
    sns.heatmap(
        heat,
        annot=annot,
        fmt='',
        cmap='coolwarm',
        center=0,
        linewidths=0.5,
        cbar_kws={'label': 'Rank-biserial effect size'}
    )

    plt.title('Lyric Feature Effects by Language')
    plt.xlabel('Language')
    plt.ylabel('Lyric feature')
    plt.tight_layout()

    save_path = OUTPUT_DIR / 'lyrics_effect_size_heatmap.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print('Saved:', save_path)
    print('* = significant after FDR correction')
else:
    print('language_tests not available. Run the within-language test cell first.')

In [ ]:
# Length-Adjusted Lexical Features by Language
adj = df_ok.copy()
for col in ['total_words','vocab_size','rare_word_count']:
    adj[col] = pd.to_numeric(adj[col], errors='coerce')

adj['hapax_token_ratio'] = np.where(
    adj['total_words'] > 0,
    adj['rare_word_count'] / adj['total_words'],
    np.nan
)
adj['vocab_length_resid'] = np.nan
adj['rare_length_resid'] = np.nan

for lang in ['English','Hindi','Chinese']:
    mask = (adj['language'] == lang) & (adj['total_words'] > 0)
    for feat, out in [('vocab_size','vocab_length_resid'),
                      ('rare_word_count','rare_length_resid')]:
        idx = mask & adj[feat].notna()
        if idx.sum() < 10:
            continue
        x = np.log1p(adj.loc[idx,'total_words'].astype(float))
        y = np.log1p(adj.loc[idx,feat].astype(float))
        slope, intercept = np.polyfit(x, y, 1)
        adj.loc[idx,out] = y - (slope * x + intercept)

ADJUSTED_FEATS = ['hapax_token_ratio','vocab_length_resid','rare_length_resid']
rows = []

for lang in ['English','Hindi','Chinese']:
    part = adj[adj['language'] == lang]
    for feat in ADJUSTED_FEATS:
        ai = pd.to_numeric(
            part.loc[part['source'] == 'AI',feat],
            errors='coerce'
        ).dropna()
        hu = pd.to_numeric(
            part.loc[part['source'] == 'Human',feat],
            errors='coerce'
        ).dropna()
        if len(ai) < 5 or len(hu) < 5:
            continue
        if pd.concat([ai,hu]).nunique() <= 1:
            continue
        u, p = stats.mannwhitneyu(ai, hu, alternative='two-sided')
        effect = (2 * u / (len(ai) * len(hu))) - 1
        rows.append({
            'language': lang,
            'feature': feat,
            'AI_mean': ai.mean(),
            'Human_mean': hu.mean(),
            'effect': effect,
            'p_value': p,
            'n_AI': len(ai),
            'n_Human': len(hu)
        })

adjusted_tests = pd.DataFrame(rows)
if not adjusted_tests.empty:
    adjusted_tests['p_FDR'] = multipletests(
        adjusted_tests['p_value'], method='fdr_bh'
    )[1]
    adjusted_tests['significant_FDR'] = adjusted_tests['p_FDR'] < 0.05

print('=== Length-Adjusted Lexical Feature Tests ===')
display(adjusted_tests.style.format({
    'AI_mean':'{:.4f}',
    'Human_mean':'{:.4f}',
    'effect':'{:.4f}',
    'p_value':'{:.3e}',
    'p_FDR':'{:.3e}'
}))

save_path = OUTPUT_DIR / 'lyrics_length_adjusted_tests.csv'
adjusted_tests.to_csv(save_path, index=False, encoding='utf-8-sig')
print('Saved:', save_path)

In [ ]:
# Length-Adjusted Lexical Effect Sizes
# Positive = higher in AI, Negative = higher in Human

if 'adjusted_tests' in globals() and not adjusted_tests.empty:
    plot_df = adjusted_tests.copy()

    labels = {
        'hapax_token_ratio':'Hapax-token ratio',
        'vocab_length_resid':'Adjusted vocabulary',
        'rare_length_resid':'Adjusted rare words',
    }

    plot_df['feature_label'] = plot_df['feature'].map(labels)

    heat = plot_df.pivot(
        index='feature_label',
        columns='language',
        values='effect'
    )[['English','Hindi','Chinese']]

    sig = plot_df.pivot(
        index='feature_label',
        columns='language',
        values='significant_FDR'
    )[['English','Hindi','Chinese']]

    annot = heat.copy().astype(str)

    for feat in heat.index:
        for lang in heat.columns:
            star = '*' if sig.loc[feat,lang] else ''
            annot.loc[feat,lang] = f'{heat.loc[feat,lang]:.2f}{star}'

    # Display Chinese as Mandarin without changing the data
    heat = heat.rename(columns={'Chinese':'Mandarin'})
    sig = sig.rename(columns={'Chinese':'Mandarin'})
    annot = annot.rename(columns={'Chinese':'Mandarin'})

    plt.figure(figsize=(8,4))

    sns.heatmap(
        heat,
        annot=annot,
        fmt='',
        cmap='coolwarm',
        center=0,
        linewidths=0.5,
        cbar_kws={'label':'Rank-biserial effect size'}
    )

    plt.title('Length-Adjusted Lexical Differences by Language')
    plt.xlabel('Language')
    plt.ylabel('Feature')
    plt.tight_layout()

    save_path = OUTPUT_DIR / 'lyrics_length_adjusted_heatmap.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print('Saved:', save_path)
    print('* = significant after FDR correction')

else:
    print('adjusted_tests not available. Run the length-adjusted test cell first.')


# Lyric Content vs Whisper ASR Diagnostics

The following analysis makes the distinction between lyric-content features and
ASR diagnostics explicit and reproducible.

In [ ]:
# Content Features vs Whisper ASR Diagnostics
# Pooled secondary analysis
CONTENT_FEATS = [
    'repetition_ratio','bigram_rep_ratio','longest_repeat_ratio',
    'ttr','vocab_size','total_words','avg_word_length','rare_word_count'
]
ASR_FEATS = [
    'whisper_avg_logprob','whisper_no_speech_ratio','whisper_lang_match'
]

rows = []
for group, feats in [('CONTENT',CONTENT_FEATS),('ASR DIAGNOSTIC',ASR_FEATS)]:
    for feat in feats:
        if feat not in df_ok.columns:
            continue
        ai = pd.to_numeric(
            df_ok.loc[df_ok['source']=='AI',feat], errors='coerce'
        ).dropna()
        hu = pd.to_numeric(
            df_ok.loc[df_ok['source']=='Human',feat], errors='coerce'
        ).dropna()
        if len(ai)<20 or len(hu)<20:
            continue
        if pd.concat([ai,hu]).nunique()<=1:
            continue

        u,p = stats.mannwhitneyu(ai,hu,alternative='two-sided')
        effect = (2*u/(len(ai)*len(hu)))-1

        rows.append({
            'group':group,
            'feature':feat,
            'AI_mean':ai.mean(),
            'Human_mean':hu.mean(),
            'rank_biserial':effect,
            'p_value':p,
            'n_AI':len(ai),
            'n_Human':len(hu)
        })

split = pd.DataFrame(rows)

if not split.empty:
    split['p_FDR'] = multipletests(
        split['p_value'], method='fdr_bh'
    )[1]
    split['significant_FDR'] = split['p_FDR']<0.05

print('=== Content Features vs Whisper ASR Diagnostics ===')
print('Pooled secondary analysis with FDR correction.')

display(split.style.format({
    'AI_mean':'{:.4f}',
    'Human_mean':'{:.4f}',
    'rank_biserial':'{:.4f}',
    'p_value':'{:.3e}',
    'p_FDR':'{:.3e}'
}))

n_content = split[
    (split['group']=='CONTENT') & split['significant_FDR']
].shape[0]
n_asr = split[
    (split['group']=='ASR DIAGNOSTIC') & split['significant_FDR']
].shape[0]

print()
print('Content significant after FDR:',n_content,'/',
      (split['group']=='CONTENT').sum())
print('ASR significant after FDR:',n_asr,'/',
      (split['group']=='ASR DIAGNOSTIC').sum())

save_path = OUTPUT_DIR/'lyrics_content_vs_asr_diagnostics.csv'
split.to_csv(save_path,index=False,encoding='utf-8-sig')
print('Saved:',save_path)

In [ ]:
# ASR Language-Match and TTR by Language and Source
# Descriptive comparison only
if 'whisper_lang_match' in df_ok.columns:
    temp = df_ok.copy()
    temp['whisper_lang_match'] = pd.to_numeric(
        temp['whisper_lang_match'], errors='coerce'
    )
    temp['ttr'] = pd.to_numeric(temp['ttr'], errors='coerce')

    print('=== Whisper Language-Match Rate (%) ===')
    lang_match = temp.pivot_table(
        index='language',
        columns='source',
        values='whisper_lang_match',
        aggfunc='mean'
    ).mul(100).round(1)
    display(lang_match)

    print('=== Mean Type-Token Ratio (TTR) ===')
    ttr_table = temp.pivot_table(
        index='language',
        columns='source',
        values='ttr',
        aggfunc='mean'
    ).round(3)
    display(ttr_table)

    print('NOTE: These descriptive tables do not establish causation.')
    print('Results may reflect ASR behaviour, recording source, vocal')
    print('separation, singing style and transcript length.')
else:
    print('whisper_lang_match column not found.')

In [ ]:
# Tokenisation Sanity Check by Language and Source
TOKEN_FEATS = ['total_words','vocab_size','ttr','avg_word_length','rare_word_count']
rows = []

for lang in ['English','Hindi','Chinese']:
    part = df_ok[df_ok['language']==lang]
    for src in ['AI','Human']:
        grp = part[part['source']==src]
        row = {'language':lang,'source':src,'n_tracks':len(grp)}
        for feat in TOKEN_FEATS:
            vals = pd.to_numeric(grp[feat],errors='coerce')
            row[feat+'_mean'] = vals.mean()
            row[feat+'_missing'] = vals.isna().sum()
        row['zero_token_tracks'] = (
            pd.to_numeric(grp['total_words'],errors='coerce').fillna(0)==0
        ).sum()
        rows.append(row)

token_check = pd.DataFrame(rows)
display(token_check.round(3))

print('Tokenisation methods:')
print('English: regex word tokenisation')
print('Hindi: Devanagari-aware regex tokenisation')
print('Chinese: Jieba word segmentation')
print()
print('Note: average token length should only be compared within the same language.')

---

# Semantic Lyric Features Multilingual Sentence Embeddings


The model is `paraphrase-multilingual-MiniLM-L12-v2`, which covers English, Hindi and
Mandarin in a single space. It was listed under "Semantic features: sentence embeddings" in
the project specification but not implemented until now.



In [ ]:
!pip install tf-keras

In [ ]:
%pip install -q sentence-transformers

In [ ]:
# Build Safe Multilingual Corpus for Lyric Embeddings

emb_source = df_ok[['song_id','source','language']].copy()

if 'full_text' in df_ok.columns:
    emb_source['full_text'] = df_ok['full_text'].fillna('').astype(str)
else:
    emb_source['full_text'] = emb_source['song_id'].apply(load_transcript)

# Remove only lines that clearly contain metadata alone
META_ONLY = re.compile(
    r'^(subscribe|official\s+(video|audio)|lyrics?\s+by.*|written\s+by.*|'
    r'produced\s+by.*|(字幕|翻译|翻譯|订阅|訂閱|作词|作詞|作曲|制作|製作|'
    r'打赏|打賞|栏目|欄目)\s*[:：\-]?\s*[\w\u4e00-\u9fff ]{0,50}|'
    r'李宗盛|明镜|明鏡|点点|點點)$',
    flags=re.IGNORECASE
)

def clean_embedding_text(text):
    lines = []
    for line in str(text).splitlines():
        line = re.sub(r'\s+',' ',line).strip()
        if not line:
            continue
        if len(line)<=100 and META_ONLY.fullmatch(line):
            continue
        lines.append(line)
    return '\n'.join(lines)

emb_source['text_for_model'] = (
    emb_source['full_text']
    .apply(clean_embedding_text)
    .str.strip()
)

emb_source['token_count'] = emb_source.apply(
    lambda row: len(tokenize_text(
        row['text_for_model'],row['language']
    )),
    axis=1
)

before = len(emb_source)
emb_source = (
    emb_source[emb_source['token_count']>=5]
    .reset_index(drop=True)
)

counts = emb_source.groupby(
    ['language','source']
).size().unstack(fill_value=0)

print('Tracks with usable transcripts:',len(emb_source),'/',before)
display(counts)

if (counts<250).any().any():
    print('WARNING: Some groups still contain fewer than 250 tracks.')
else:
    print('Corpus coverage looks acceptable.')

index_path = OUTPUT_DIR/'lyrics_embedding_index.csv'
emb_source[
    ['song_id','source','language','token_count']
].to_csv(index_path,index=False,encoding='utf-8-sig')

print('Saved:',index_path)

In [ ]:
# Audit Tracks Excluded from Lyric Embeddings
used_ids = set(emb_source['song_id'].astype(str))

embedding_excluded = df_ok[
    ~df_ok['song_id'].astype(str).isin(used_ids)
][['song_id','language','source','total_words','transcript_ok']].copy()

display(embedding_excluded)

save_path = OUTPUT_DIR/'lyrics_embedding_excluded_tracks.csv'
embedding_excluded.to_csv(save_path,index=False,encoding='utf-8-sig')

print('Excluded tracks:',len(embedding_excluded))
print('Saved:',save_path)

In [ ]:
# Multilingual Lyric Embeddings with Token Chunking  CPU

import torch
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# paraphrase-multilingual-MiniLM-L12-v2 maps text to a 384-dimensional vector
# in a shared cross-lingual space, so English, Hindi and Mandarin lyrics all
# end up in the same coordinate system.
# Long transcripts are split into 126-token chunks (model max is 128);
# chunk vectors are averaged and L2-normalised to give one vector per track.
EMB_MODEL_ID = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
EMB_DEVICE = 'cpu'
EMB_BATCH_SIZE = 16

print('Loading:',EMB_MODEL_ID)
print('Device:',EMB_DEVICE)

embedder = SentenceTransformer(
    EMB_MODEL_ID,
    device=EMB_DEVICE
)

tokenizer = embedder.tokenizer
chunk_size = embedder.max_seq_length-2  # -2 reserves slots for [CLS] and [SEP] special tokens

print('Model maximum tokens:',embedder.max_seq_length)
print('Chunk content tokens:',chunk_size)

chunks,owners = [],[]

for i,text in enumerate(
    emb_source['text_for_model'].fillna('').astype(str)
):
    ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    pieces = [
        ids[j:j+chunk_size]
        for j in range(0,len(ids),chunk_size)
    ]

    for piece in pieces:
        chunk = tokenizer.decode(
            piece,
            skip_special_tokens=True
        ).strip()

        if chunk:
            chunks.append(chunk)
            owners.append(i)

print('Tracks:',len(emb_source))
print('Total chunks:',len(chunks))

chunk_matrix = embedder.encode(
    chunks,
    batch_size=EMB_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    device=EMB_DEVICE
).astype(np.float32)

owners = np.asarray(owners,dtype=int)

emb_matrix = np.zeros(
    (len(emb_source),chunk_matrix.shape[1]),
    dtype=np.float32
)

np.add.at(emb_matrix,owners,chunk_matrix)  # accumulate chunk vectors into their track row

chunk_counts = np.bincount(
    owners,
    minlength=len(emb_source)
).astype(np.float32)

assert np.all(chunk_counts>0),'A track received no embedding chunks.'

emb_matrix /= chunk_counts[:,None]  # divide by chunk count to get mean vector per track

norms = np.linalg.norm(
    emb_matrix,
    axis=1,
    keepdims=True
)

emb_matrix /= np.clip(norms,1e-12,None)  # L2-normalise: all vectors end up on the unit sphere

emb_meta = emb_source[
    ['song_id','source','language','token_count']
].reset_index(drop=True)

emb_meta.insert(
    0,
    'embedding_row',
    np.arange(len(emb_meta))
)

emb_meta['n_chunks'] = chunk_counts.astype(int)

np.save(
    OUTPUT_DIR/'lyrics_embeddings.npy',
    emb_matrix
)

emb_meta.to_csv(
    OUTPUT_DIR/'lyrics_embedding_metadata.csv',
    index=False,
    encoding='utf-8-sig'
)

EMB_COLS = [
    f'emb_{i}'
    for i in range(emb_matrix.shape[1])
]

emb_df = pd.concat([
    emb_meta[
        ['song_id','source','language']
    ].reset_index(drop=True),
    pd.DataFrame(
        emb_matrix,
        columns=EMB_COLS
    )
],axis=1)

y_emb = (
    emb_df['source']=='AI'
).astype(int).values

assert len(emb_matrix)==len(emb_meta)==len(emb_df)
assert emb_meta['song_id'].is_unique

print('Embedding matrix:',emb_matrix.shape)
print('Class balance:')
print(emb_meta['source'].value_counts())
print('Chunks per track:')
print(emb_meta['n_chunks'].describe().round(2))
print('Saved: outputs/lyrics_embeddings.npy')
print('Saved: outputs/lyrics_embedding_metadata.csv')

In [ ]:
# Lyric Embeddings: 5-Fold Out-of-Fold Evaluation
# Folds balanced by joint language and source

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,f1_score,roc_auc_score
from sklearn.base import clone

EMB_SEED = 42

def emb_pipe(clf):
    return Pipeline([
        ('scaler',StandardScaler()),
        ('clf',clf)
    ])

EMB_MODELS = {
    'Logistic Regression':emb_pipe(
        LogisticRegression(
            max_iter=3000,
            class_weight='balanced',
            random_state=EMB_SEED
        )
    ),
    'SVM (RBF)':emb_pipe(
        SVC(
            kernel='rbf',
            class_weight='balanced',
            random_state=EMB_SEED
        )
    )
}

X_emb = emb_df[EMB_COLS].to_numpy(dtype=np.float32)
y_emb = (emb_df['source']=='AI').astype(int).to_numpy()
strat_key = (
    emb_df['language'].astype(str)+'_'+emb_df['source'].astype(str)
).to_numpy()

cv = StratifiedKFold(5,shuffle=True,random_state=EMB_SEED)
splits = list(cv.split(X_emb,strat_key))

result_rows = []
oof_frames = []

for name,model in EMB_MODELS.items():
    oof_pred = np.zeros(len(y_emb),dtype=int)
    oof_score = np.zeros(len(y_emb),dtype=float)
    fold_ids = np.zeros(len(y_emb),dtype=int)

    for fold,(train_idx,test_idx) in enumerate(splits,1):
        fitted = clone(model)
        fitted.fit(X_emb[train_idx],y_emb[train_idx])

        oof_pred[test_idx] = fitted.predict(X_emb[test_idx])
        oof_score[test_idx] = fitted.decision_function(X_emb[test_idx])
        fold_ids[test_idx] = fold

    result_rows.append({
        'model':name,
        'accuracy':accuracy_score(y_emb,oof_pred),
        'macro_f1':f1_score(y_emb,oof_pred,average='macro'),
        'auroc':roc_auc_score(y_emb,oof_score),
        'n_tracks':len(y_emb)
    })

    temp = emb_df[['song_id','source','language']].copy()
    temp['model'] = name
    temp['fold'] = fold_ids
    temp['y_true'] = y_emb
    temp['y_pred'] = oof_pred
    temp['score'] = oof_score
    oof_frames.append(temp)

emb_indist = pd.DataFrame(result_rows)
emb_oof = pd.concat(oof_frames,ignore_index=True)

print('=== Lyric Embeddings: 5-Fold OOF Evaluation ===')
print('Folds stratified by joint language and source.')
display(emb_indist.style.format({
    'accuracy':'{:.4f}',
    'macro_f1':'{:.4f}',
    'auroc':'{:.4f}'
}))

emb_indist.to_csv(
    OUTPUT_DIR/'lyrics_embedding_cv_results.csv',
    index=False,encoding='utf-8-sig'
)
emb_oof.to_csv(
    OUTPUT_DIR/'lyrics_embedding_oof_predictions.csv',
    index=False,encoding='utf-8-sig'
)

print('Saved: outputs/lyrics_embedding_cv_results.csv')
print('Saved: outputs/lyrics_embedding_oof_predictions.csv')

In [ ]:
# Lyric Embeddings: Cross-Language Generalisation
# Same model used for in-distribution and language holdout

from sklearn.base import clone
from sklearn.metrics import accuracy_score,f1_score,roc_auc_score

xlang_rows = []

for held in ['English','Hindi','Chinese']:
    train_mask = emb_df['language']!=held
    test_mask = emb_df['language']==held
    train_idx = np.where(train_mask)[0]
    test_idx = np.where(test_mask)[0]

    if len(test_idx)<20 or len(train_idx)<50:
        continue

    for name,model in EMB_MODELS.items():
        fitted = clone(model)
        fitted.fit(X_emb[train_idx],y_emb[train_idx])

        pred = fitted.predict(X_emb[test_idx])
        score = fitted.decision_function(X_emb[test_idx])

        xlang_rows.append({
            'held_out':held,
            'model':name,
            'n_train':len(train_idx),
            'n_test':len(test_idx),
            'accuracy':accuracy_score(y_emb[test_idx],pred),
            'macro_f1':f1_score(y_emb[test_idx],pred,average='macro'),
            'auroc':roc_auc_score(y_emb[test_idx],score)
        })

emb_xlang = pd.DataFrame(xlang_rows)

print('=== Lyric Embeddings: Cross-Language Generalisation ===')
display(emb_xlang.style.format({
    'accuracy':'{:.4f}',
    'macro_f1':'{:.4f}',
    'auroc':'{:.4f}'
}))

xlang_mean = (
    emb_xlang.groupby('model')[['accuracy','macro_f1','auroc']]
    .mean()
    .reset_index()
    .rename(columns={
        'accuracy':'crosslang_accuracy',
        'macro_f1':'crosslang_macro_f1',
        'auroc':'crosslang_auroc'
    })
)

comparison = emb_indist[
    ['model','accuracy','macro_f1','auroc']
].merge(xlang_mean,on='model',how='inner')

comparison = comparison.rename(columns={
    'accuracy':'indist_accuracy',
    'macro_f1':'indist_macro_f1',
    'auroc':'indist_auroc'
})

comparison['f1_drop'] = (
    comparison['indist_macro_f1']-
    comparison['crosslang_macro_f1']
)
comparison['auroc_drop'] = (
    comparison['indist_auroc']-
    comparison['crosslang_auroc']
)

print('\n=== Same-Model In-Distribution vs Cross-Language ===')
display(comparison.style.format({
    'indist_accuracy':'{:.4f}',
    'indist_macro_f1':'{:.4f}',
    'indist_auroc':'{:.4f}',
    'crosslang_accuracy':'{:.4f}',
    'crosslang_macro_f1':'{:.4f}',
    'crosslang_auroc':'{:.4f}',
    'f1_drop':'{:.4f}',
    'auroc_drop':'{:.4f}'
}))

emb_xlang.to_csv(
    OUTPUT_DIR/'lyrics_embedding_crosslang.csv',
    index=False,encoding='utf-8-sig'
)
comparison.to_csv(
    OUTPUT_DIR/'lyrics_embedding_crosslang_comparison.csv',
    index=False,encoding='utf-8-sig'
)

print('Saved: outputs/lyrics_embedding_crosslang.csv')
print('Saved: outputs/lyrics_embedding_crosslang_comparison.csv')

In [ ]:
# Lyric Embedding Evaluation Summary

if 'comparison' in globals() and not comparison.empty:
    best = comparison.sort_values(
        'crosslang_macro_f1',ascending=False
    ).iloc[0]

    best_model = best['model']
    held = emb_xlang[emb_xlang['model']==best_model].copy()

    print('='*68)
    print('LYRIC EMBEDDING EVALUATION SUMMARY')
    print('='*68)
    print('Best same-model result:',best_model)
    print('In-distribution macro-F1 : %.4f'%best['indist_macro_f1'])
    print('Cross-language mean F1   : %.4f'%best['crosslang_macro_f1'])
    print('Macro-F1 drop            : %.4f'%best['f1_drop'])
    print('In-distribution AUROC    : %.4f'%best['indist_auroc'])
    print('Cross-language mean AUROC: %.4f'%best['crosslang_auroc'])
    print('AUROC drop               : %.4f'%best['auroc_drop'])
    print()

    print('Held-out language results:')
    display(held[[
        'held_out','n_test','accuracy','macro_f1','auroc'
    ]].style.format({
        'accuracy':'{:.4f}',
        'macro_f1':'{:.4f}',
        'auroc':'{:.4f}'
    }))

    print('Interpretation:')
    print('Embedding performance must be interpreted from both the')
    print('in-distribution and individual held-out-language results.')
    print('A lower cross-language score indicates limited transfer,')
    print('but does not prove that lyric content contains no signal.')
    print('Final comparison with handcrafted lyric and audio features')
    print('must use the same splits and evaluation procedure in NB3.')
    print('='*68)
else:
    print('Run the embedding CV and cross-language cells first.')

In [ ]:
# Lyric Embedding Performance Heatmap
if ('emb_xlang' in globals() and not emb_xlang.empty and
        'comparison' in globals() and not comparison.empty):

    heat = emb_xlang.pivot(
        index='model',columns='held_out',values='macro_f1'
    )

    for lang in ['English','Hindi','Chinese']:
        if lang not in heat.columns:
            heat[lang] = np.nan

    heat = heat[['English','Hindi','Chinese']]
    heat['Mean holdout'] = heat.mean(axis=1)

    indist = comparison.set_index('model')['indist_macro_f1']
    heat.insert(0,'In-distribution',indist.reindex(heat.index))

    plt.figure(figsize=(9,3.5))
    sns.heatmap(
        heat,annot=True,fmt='.3f',
        cmap='YlGnBu',vmin=0.5,vmax=1.0,
        linewidths=0.5,
        cbar_kws={'label':'Macro-F1'}
    )
    plt.title('Lyric Embedding Performance by Evaluation Condition',
              fontweight='bold')
    plt.xlabel('Evaluation condition')
    plt.ylabel('Model')
    plt.xticks(rotation=20,ha='right')
    plt.tight_layout()

    png_path = OUTPUT_DIR/'lyrics_embedding_performance_heatmap.png'
    pdf_path = OUTPUT_DIR/'lyrics_embedding_performance_heatmap.pdf'
    plt.savefig(png_path,dpi=300,bbox_inches='tight')
    plt.savefig(pdf_path,bbox_inches='tight')
    plt.show()

    print('Saved:',png_path)
    print('Saved:',pdf_path)
else:
    print('Run the embedding CV and cross-language cells first.')